# 4.3 — Dates, Strings, and Reduction

**Chapter 4, sections 4.3.1 to 4.3.3**, and the starting point for **Exercise 4**.

**The question this notebook answers:** three of the chapter's transformation techniques make
the same argument from different directions. A built-in function knows what a value *is*; the
string surgery that substitutes for one is load-bearing on positions that nothing guarantees.

1. **Dates and times** — the course's own five-`substring` listing, and what replaces it.
2. **Strings** — the accessors and combiners whose behaviour differs in ways that only show up
   on the second dataset.
3. **Filtering and projection** — the cheapest transformation there is, and the two ways
   PySpark's syntax bites.

**Data.** The taxi file, read twice: once with `pickup_datetime` declared a **string**, which is
how it arrives and what the anti-pattern needs, and once as a **timestamp**.

Runs on a laptop in about a minute and a half.

In [1]:
# --- CS-777 session setup ------------------------------------------------
import os, tempfile, time, logging
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType,
                               IntegerType, DoubleType, TimestampType)

DATA = os.environ.get("CS777_DATA", "../data")
SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))
os.makedirs(SCRATCH, exist_ok=True)

spark = (SparkSession.builder
         .appName("CS777-4.3")
         .master("local[*]")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
for _lg in ("SQLQueryContextLogger", "DataFrameQueryContextLogger"):
    logging.getLogger(_lg).setLevel(logging.CRITICAL)

TAXI = f"{DATA}/taxi-data-sorted-small.csv.bz2"

names = ["medallion", "hack_license", "pickup_datetime", "dropoff_datetime",
         "trip_time", "trip_distance", "pickup_longitude", "pickup_latitude",
         "dropoff_longitude", "dropoff_latitude", "payment_type", "fare_amount",
         "surcharge", "mta_tax", "tip_amount", "tolls_amount", "total_amount"]

def taxi_schema(datetimes_as):
    """The seventeen columns, with the two datetime fields as string or as timestamp."""
    types = {"medallion": StringType(), "hack_license": StringType(),
             "pickup_datetime": datetimes_as, "dropoff_datetime": datetimes_as,
             "trip_time": IntegerType(), "trip_distance": DoubleType(),
             "payment_type": StringType()}
    return StructType([StructField(n, types.get(n, DoubleType()), True) for n in names])

def read(datetimes_as):
    """All seventeen columns, exactly as the file carries them."""
    return spark.read.schema(taxi_schema(datetimes_as)).option("header", "false").csv(TAXI)

print("Spark", spark.version,
      "| session time zone:", spark.conf.get("spark.sql.session.timeZone"))

Spark 4.2.0 | session time zone: Europe/Skopje


## 1. Dates and times

### The anti-pattern, reproduced exactly

The chapter reproduces the course notebook's five `substring` calls unmodified, comments
included, because it is the pattern this section exists to retire. Here it is, running.

In [2]:
raw = read(StringType()).cache()          # pickup_datetime is text, as it arrives
print(f"{raw.count():,} rows, pickup_datetime declared as:",
      dict(raw.dtypes)["pickup_datetime"])

from pyspark.sql.functions import col, substring
# Note: Please note that the position is not zero based, but 1 based index.
# Create a new columns wiht the year, month, day, hour and minute of the taxi trip
df = raw.withColumn('year', substring('pickup_datetime', 1,4))\
    .withColumn('month', substring('pickup_datetime', 6,2))\
    .withColumn('day', substring('pickup_datetime', 9,2))\
    .withColumn('hour', substring('pickup_datetime', 12,2))\
    .withColumn('minute', substring('pickup_datetime', 15,2))
df.select('pickup_datetime', 'year', 'month', 'day', 'hour', 'minute').show(5, truncate=False)
print({c: t for c, t in df.dtypes if c in ("year", "month", "day", "hour", "minute")})

1,999,999 rows, pickup_datetime declared as: string
+-------------------+----+-----+---+----+------+
|pickup_datetime    |year|month|day|hour|minute|
+-------------------+----+-----+---+----+------+
|2013-01-01 00:00:00|2013|01   |01 |00  |00    |
|2013-01-01 00:02:00|2013|01   |01 |00  |02    |
|2013-01-01 00:01:00|2013|01   |01 |00  |01    |
|2013-01-01 00:01:00|2013|01   |01 |00  |01    |
|2013-01-01 00:01:00|2013|01   |01 |00  |01    |
+-------------------+----+-----+---+----+------+
only showing top 5 rows
{'year': 'string', 'month': 'string', 'day': 'string', 'hour': 'string', 'minute': 'string'}


The first thing to say about this code is that **it works**, which is why it survives in so many
pipelines. Its defects arrive in order of increasing severity, and each has a concrete trigger.

In [3]:
# Defect 1 -- types. Every derived field is a string, so it sorts like one.
months = spark.createDataFrame([("2013-02-01 00:00:00",), ("2013-10-01 00:00:00",),
                                ("2013-01-01 00:00:00",)], ["pickup_datetime"])
print("string months, sorted:",
      [r[0] for r in months.select(substring("pickup_datetime", 6, 2).alias("m"))
                           .orderBy("m").collect()])
print("string months > '2' :",
      months.where(substring("pickup_datetime", 6, 2) > "2").count(), "of 3 rows"
      "   <- '10' is not greater than '2' as text")

string months, sorted: ['01', '02', '10']
string months > '2' : 0 of 3 rows   <- '10' is not greater than '2' as text


In [4]:
# Defect 2 -- robustness. One extra character, and every field shifts.
shifted = spark.createDataFrame([("2013-01-05 14:30:00",),      # well formed
                                 ("2013-01-05  14:30:00",),     # one extra space
                                 ("05/01/2013 14:30:00",)],     # another source's format
                                ["pickup_datetime"])
shifted.select("pickup_datetime",
               substring("pickup_datetime", 1, 4).alias("year"),
               substring("pickup_datetime", 6, 2).alias("month"),
               substring("pickup_datetime", 12, 2).alias("hour")).show(truncate=False)

+--------------------+----+-----+----+
|pickup_datetime     |year|month|hour|
+--------------------+----+-----+----+
|2013-01-05 14:30:00 |2013|01   |14  |
|2013-01-05  14:30:00|2013|01   | 1  |
|05/01/2013 14:30:00 |05/0|/2   |14  |
+--------------------+----+-----+----+



Nothing failed. The second row reports the hour as `" 1"` instead of `14`, and the third reports
the year `05/0` and the month `/2`. A pipeline built on character positions produces
answers of exactly this kind, and no test that checks for exceptions will ever see them.

**Defect 3 is decisive**: no combination of substrings can compute the duration between two
timestamps, the day of the week, or the end of a month, because the strings do not know they are
times.

### One conversion, and every temporal question becomes a function call

In [5]:
ts = (raw
      .withColumn("pickup_ts",  F.to_timestamp("pickup_datetime",  "yyyy-MM-dd HH:mm:ss"))
      .withColumn("dropoff_ts", F.to_timestamp("dropoff_datetime", "yyyy-MM-dd HH:mm:ss")))

ts = (ts.withColumn("year",    F.year("pickup_ts"))
        .withColumn("hour",    F.hour("pickup_ts"))
        .withColumn("weekday", F.dayofweek("pickup_ts"))    # 1 = Sunday ... 7 = Saturday
        .withColumn("trip_minutes",
                    (F.unix_timestamp("dropoff_ts")
                     - F.unix_timestamp("pickup_ts")) / 60)).cache()

ts.select("pickup_ts", "year", "hour", "weekday", "trip_minutes",
          F.date_format("pickup_ts", "EEEE").alias("weekday_name")).show(5)
print({c: t for c, t in ts.dtypes if c in ("pickup_ts", "year", "hour", "trip_minutes")})

+-------------------+----+----+-------+------------+------------+
|          pickup_ts|year|hour|weekday|trip_minutes|weekday_name|
+-------------------+----+----+-------+------------+------------+
|2013-01-01 00:00:00|2013|   0|      3|         2.0|     Tuesday|
|2013-01-01 00:02:00|2013|   0|      3|         0.0|     Tuesday|
|2013-01-01 00:01:00|2013|   0|      3|         2.0|     Tuesday|
|2013-01-01 00:01:00|2013|   0|      3|         2.0|     Tuesday|
|2013-01-01 00:01:00|2013|   0|      3|         2.0|     Tuesday|
+-------------------+----+----+-------+------------+------------+
only showing top 5 rows
{'pickup_ts': 'timestamp', 'year': 'int', 'hour': 'int', 'trip_minutes': 'double'}


In [6]:
# The two forms agree on well-formed input -- an assertion, not a sentence.
check = (ts.withColumn("substr_year",  substring("pickup_datetime", 1, 4).cast("int"))
           .withColumn("substr_hour",  substring("pickup_datetime", 12, 2).cast("int"))
           .where((F.col("year") != F.col("substr_year")) |
                  (F.col("hour") != F.col("substr_hour"))).count())
assert check == 0, f"{check} rows disagree"
print("substring and F.year/F.hour agree on all", f"{ts.count():,}",
      "rows of this file -- and the file is well formed, which is the only reason.")

# What only the timestamp can answer: duration, and the calendar.
ts.groupBy(F.date_format("pickup_ts", "EEEE").alias("weekday")).agg(
    F.count("*").alias("trips"),
    F.round(F.avg("trip_minutes"), 1).alias("avg_minutes"),
    F.round(F.avg("trip_distance"), 2).alias("avg_miles")
).orderBy(F.desc("trips")).show()

substring and F.year/F.hour agree on all 1,999,999 rows of this file -- and the file is well formed, which is the only reason.


+---------+------+-----------+---------+
|  weekday| trips|avg_minutes|avg_miles|
+---------+------+-----------+---------+
|  Tuesday|454708|       11.3|     3.03|
|Wednesday|445541|       11.7|     2.89|
| Thursday|259894|       11.3|     2.84|
|   Friday|237190|       11.1|      2.7|
| Saturday|226405|       10.6|     2.77|
|   Monday|216473|       11.2|     2.85|
|   Sunday|159788|       10.9|     3.08|
+---------+------+-----------+---------+



### The pattern language has two rules worth memorizing

In [7]:
one = spark.createDataFrame([("2013-10-05 14:30:00",)], ["s"])
one.select(
    F.to_timestamp("s", "yyyy-MM-dd HH:mm:ss").alias("correct"),
    F.to_timestamp("s", "yyyy-dd-MM HH:mm:ss").alias("MM_and_dd_transposed"),
    F.date_format(F.to_timestamp("s", "yyyy-MM-dd HH:mm:ss"), "MM").alias("MM_is_month"),
    F.date_format(F.to_timestamp("s", "yyyy-MM-dd HH:mm:ss"), "mm").alias("mm_is_minute"),
).show(truncate=False)

+-------------------+--------------------+-----------+------------+
|correct            |MM_and_dd_transposed|MM_is_month|mm_is_minute|
+-------------------+--------------------+-----------+------------+
|2013-10-05 14:30:00|2013-05-10 14:30:00 |10         |30          |
+-------------------+--------------------+-----------+------------+



In [8]:
# Rule 2: parsing is exact, and failure is an error under ANSI mode -- not a null.
def outcome(label, fn):
    try:
        print(f"{label:46s} -> {fn()}")
    except Exception as e:
        print(f"{label:46s} -> {type(e).__name__}: "
              f"{getattr(e, 'getCondition', lambda: '?')()}")

spark.sparkContext.setLogLevel("FATAL")          # the failures below are deliberate
print("legacy parser policy:", spark.conf.get("spark.sql.legacy.timeParserPolicy"))
outcome("to_timestamp('2013-1-5', 'yyyy-MM-dd')",
        lambda: spark.sql("SELECT to_timestamp('2013-1-5','yyyy-MM-dd')").first()[0])
outcome("to_timestamp('not a date', 'yyyy-MM-dd')",
        lambda: spark.sql("SELECT to_timestamp('not a date','yyyy-MM-dd')").first()[0])
outcome("try_to_timestamp('not a date', 'yyyy-MM-dd')",
        lambda: spark.sql("SELECT try_to_timestamp('not a date','yyyy-MM-dd')").first()[0])
outcome("date_format(ts, 'EEEEE')  (five pattern letters)",
        lambda: spark.sql("SELECT date_format(timestamp'2013-01-01', 'EEEEE')").first()[0])
spark.sparkContext.setLogLevel("ERROR")

legacy parser policy: CORRECTED
to_timestamp('2013-1-5', 'yyyy-MM-dd')         -> DateTimeException: CANNOT_PARSE_TIMESTAMP
to_timestamp('not a date', 'yyyy-MM-dd')       -> DateTimeException: CANNOT_PARSE_TIMESTAMP
try_to_timestamp('not a date', 'yyyy-MM-dd')   -> None
date_format(ts, 'EEEEE')  (five pattern letters) -> SparkUpgradeException: INCONSISTENT_BEHAVIOR_CROSS_VERSION.DATETIME_PATTERN_RECOGNITION


Parsing really is exact: `'2013-1-5'` is **rejected** against `yyyy-MM-dd`, because `MM`
declares two digits and the string supplies one. A source that drops its leading zeros is
therefore a source that fails, loudly, which is the better of the two available outcomes. An
unparseable timestamp raises `CANNOT_PARSE_TIMESTAMP` under ANSI mode rather than returning
null; `try_to_timestamp` is the tolerant form, with the same accounting obligation as `try_cast`
in [4.2](04.02%20Cleaning%20the%20Taxi%20Data.ipynb). And a text field of five or more pattern
letters is refused outright, with an error that names the version whose behaviour changed.

### Truncate to group, format only to display

The distinction inside the formatting group decides the correctness of a grouped query.

In [9]:
quarters = spark.createDataFrame(
    [("2013-01-15",), ("2013-02-15",), ("2013-04-15",), ("2013-10-15",), ("2013-12-15",)],
    ["d"]).withColumn("ts", F.to_timestamp("d", "yyyy-MM-dd"))

print("grouped by date_format(..., 'MMMM') -- a string, sorted alphabetically:")
(quarters.groupBy(F.date_format("ts", "MMMM").alias("month"))
 .count().orderBy("month").show())

print("grouped by date_trunc('month', ...) -- a timestamp, which still sorts as time:")
(quarters.groupBy(F.date_trunc("month", "ts").alias("month"))
 .count().orderBy("month").show(truncate=False))

grouped by date_format(..., 'MMMM') -- a string, sorted alphabetically:


+--------+-----+
|   month|count|
+--------+-----+
|   April|    1|
|December|    1|
|February|    1|
| January|    1|
| October|    1|
+--------+-----+

grouped by date_trunc('month', ...) -- a timestamp, which still sorts as time:


+-------------------+-----+
|month              |count|
+-------------------+-----+
|2013-01-01 00:00:00|1    |
|2013-02-01 00:00:00|1    |
|2013-04-01 00:00:00|1    |
|2013-10-01 00:00:00|1    |
|2013-12-01 00:00:00|1    |
+-------------------+-----+



April first. `date_format` returns twelve strings and the sort is alphabetical; `date_trunc`
returns a timestamp that still sorts, compares and subtracts as time. The rule: **truncate to
group, and format only to display, at the last step.**

### One silent parameter governs all of the above

In [10]:
hours = ts.select("pickup_ts").cache()
print(f"{'session time zone':22s} {'hour 0':>9s} {'hour 1':>9s} {'hour 2':>9s}   busiest hour")
for zone in ["UTC", "America/New_York", "Europe/Berlin"]:
    spark.conf.set("spark.sql.session.timeZone", zone)
    counts = {r["h"]: r["n"] for r in
              hours.groupBy(F.hour("pickup_ts").alias("h")).count()
                   .withColumnRenamed("count", "n").collect()}
    busiest = max(counts, key=counts.get)
    print(f"{zone:22s} {counts[0]:>9,} {counts[1]:>9,} {counts[2]:>9,}   {busiest:02d}:00")

spark.conf.set("spark.sql.session.timeZone", "UTC")      # the remedy, one line, at the top
print("\nset explicitly:", spark.conf.get("spark.sql.session.timeZone"))

session time zone         hour 0    hour 1    hour 2   busiest hour


UTC                       47,585    38,488    30,002   17:00
America/New_York          43,037    76,217    96,432   12:00


Europe/Berlin             66,378    47,585    38,488   18:00

set explicitly: UTC


Nothing failed here either. The three rows are the same query over the same data, and they
disagree about which hour of the day New York's taxis are busiest — because
`spark.sql.session.timeZone` defaults to the local zone of the machine, and every function that
moves between strings, timestamps and epoch seconds consults it. This is the answer to
**Exercise 4(c)**: the parameter is `spark.sql.session.timeZone`, the remedy is
`spark.conf.set("spark.sql.session.timeZone", "UTC")`, and it belongs at the top of the job,
with conversion to a local zone performed only at presentation time.

## 2. String manipulation

Every function in this section is a built-in that Catalyst compiles into generated code, so
there is no performance argument for writing a Python function that duplicates one — the subject
of [4.6](04.06%20UDF%20Performance%20Hierarchy.ipynb). What matters here is that several of them
behave differently in ways a single-row example never reveals.

In [11]:
codes = spark.createDataFrame(
    [("A-B-C", "CSH", "7"), ("X-Y", None, "42"), ("Z", "CRD", "310")],
    ["route_code", "payment_type", "zone"])

parts = (codes
         .withColumn("segments",  F.split("route_code", "-"))
         .withColumn("first_seg", F.col("segments").getItem(0))       # counts from ZERO
         .withColumn("first_at1", F.element_at("segments", 1))        # counts from ONE
         .withColumn("last_seg",  F.element_at("segments", -1))       # from the end
         .withColumn("concat",    F.concat("payment_type", F.lit("/"), "route_code"))
         .withColumn("concat_ws", F.concat_ws("/", "payment_type", "route_code"))
         .withColumn("zone_padded", F.lpad("zone", 5, "0")))

parts.select("route_code", "first_seg", "first_at1", "last_seg",
             "payment_type", "concat", "concat_ws", "zone_padded").show(truncate=False)

+----------+---------+---------+--------+------------+---------+---------+-----------+
|route_code|first_seg|first_at1|last_seg|payment_type|concat   |concat_ws|zone_padded|
+----------+---------+---------+--------+------------+---------+---------+-----------+
|A-B-C     |A        |A        |C       |CSH         |CSH/A-B-C|CSH/A-B-C|00007      |
|X-Y       |X        |X        |Y       |NULL        |NULL     |X-Y      |00042      |
|Z         |Z        |Z        |Z       |CRD         |CRD/Z    |CRD/Z    |00310      |
+----------+---------+---------+--------+------------+---------+---------+-----------+



Three behavioural facts, all visible above and all impossible to see on one well-formed row:

* **Indexing differs between the accessors.** `getItem(0)` and `element_at(..., 1)` return the
  same segment. Writing `element_at(..., 0)` is an error rather than the first element.
* **Null handling differs between the combiners.** Row two has a null `payment_type`: `concat`
  propagates the null and returns null for the whole expression, while `concat_ws` skips it and
  joins what remains.
* **`lpad` restores the leading zeros a numeric detour stripped**, but only because the correct
  width is known. It is a repair of last resort, not a substitute for declaring an
  identifier-like column as text in the first place.

In [12]:
# levenshtein and soundex produce candidates, not decisions.
spellings = spark.createDataFrame(
    [("CASH", "CSH"), ("CASH", "CASE"), ("Smith", "Smyth"), ("CREDIT", "CRD")],
    ["a", "b"])
spellings.select("a", "b",
                 F.levenshtein("a", "b").alias("edit_distance"),
                 F.soundex("a").alias("soundex_a"),
                 F.soundex("b").alias("soundex_b"),
                 (F.soundex("a") == F.soundex("b")).alias("same_sound")).show()

+------+-----+-------------+---------+---------+----------+
|     a|    b|edit_distance|soundex_a|soundex_b|same_sound|
+------+-----+-------------+---------+---------+----------+
|  CASH|  CSH|            1|     C200|     C000|     false|
|  CASH| CASE|            1|     C200|     C200|      true|
| Smith|Smyth|            1|     S530|     S530|      true|
|CREDIT|  CRD|            3|     C633|     C630|     false|
+------+-----+-------------+---------+---------+----------+



A distance of one separates both a typo from its correction and `"Smith"` from `"Smyth"`, who
may be different people. Their sound use is to *generate candidate pairs* for the mapping table
of [4.2](04.02%20Cleaning%20the%20Taxi%20Data.ipynb), where a person confirms them.

## 3. Filtering, selection, and projection

The syntax is the simplest in the chapter; the cost model is what deserves attention.
Reduction is the cheapest transformation there is, because it shrinks every operation that
follows.

### The course notebook's row validation, twice

In [13]:
# The notebook works in the RDD API and validates records with a Python predicate.
def isfloat(value):
    try:
        float(value)
        return True
    except ValueError:
        return False

def correctRows(p):
    if(len(p)==17):
        if(isfloat(p[5]) and isfloat(p[11])):
            if(float(p[4])> 60 and float(p[5])>0.10 and
               float(p[11])> 0.10 and float(p[16])> 0.10):
                return p

t0 = time.time()
rdd_valid = (spark.sparkContext.textFile(TAXI)
             .map(lambda line: line.split(","))
             .filter(correctRows)
             .count())
rdd_seconds = time.time() - t0
print(f"RDD + Python predicate : {rdd_valid:,} rows kept in {rdd_seconds:5.1f}s")

RDD + Python predicate : 1,965,346 rows kept in   3.3s


In [14]:
# The same minima, as a column expression over a schema the DataFrame already enforces.
# Neither side is cached: both read and decompress the same bz2 file on every run, which is
# the only way the two numbers are about the two implementations rather than about a cache.
typed = read(TimestampType())

t0 = time.time()
valid = typed.filter(
    (F.col("trip_time")     > 60) &
    (F.col("trip_distance") > 0.10) &
    (F.col("fare_amount")   > 0.10) &
    (F.col("total_amount")  > 0.10))
df_valid = valid.count()
df_seconds = time.time() - t0
print(f"DataFrame column filter: {df_valid:,} rows kept in {df_seconds:5.1f}s")
assert df_valid == rdd_valid, "the two predicates do not agree"
print(f"same answer, {rdd_seconds / df_seconds:.1f}x faster on this machine "
      "(the ratio is machine-dependent; the sign of the difference is not)")
print("part of that margin is the Python boundary and part is the four columns the")
print("DataFrame reader never parsed -- two halves of the same argument.")

trips = valid.select("pickup_datetime", "trip_distance", "fare_amount", "total_amount")
print(f"\nthe select prunes {len(typed.columns) - len(trips.columns)} of "
      f"{len(typed.columns)} columns out of everything downstream")

DataFrame column filter: 1,965,346 rows kept in   1.7s
same answer, 2.0x faster on this machine (the ratio is machine-dependent; the sign of the difference is not)
part of that margin is the Python boundary and part is the four columns the
DataFrame reader never parsed -- two halves of the same argument.

the select prunes 13 of 17 columns out of everything downstream


The two versions state the same minima and the difference between them is not style.
`correctRows` is Python: it runs in a Python worker process, row by row, invisible to the
optimizer, and its length check and float probes exist only because a schemaless RDD line might
contain anything. The column expression is a *description*: checked against a schema the
DataFrame already enforces, evaluated inside the JVM, and reorderable by the planner.

### The plan is where the reduction becomes visible

In [15]:
# Parquet, so that a filter can actually be pushed into the reader.
PARQ = os.path.join(SCRATCH, "ch04-taxi-parquet")
if not os.path.exists(PARQ):
    typed.write.mode("overwrite").parquet(PARQ)

import io, contextlib

def scan_details(dataframe):
    """The PushedFilters and ReadSchema lines of a plan's file scan.

    `explain(mode="formatted")` is used rather than the plan string, because the plan string
    truncates long lists with an ellipsis -- which is how the filters that matter go missing.
    """
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        dataframe.explain(mode="formatted")
    for line in buffer.getvalue().split("\n"):
        if line.strip().startswith(("PushedFilters", "ReadSchema")):
            print("  ", line.strip()[:260])

scan_details(spark.read.parquet(PARQ)
             .where((F.col("trip_distance") > 0.10) & (F.col("fare_amount") > 0.10))
             .select("pickup_datetime", "trip_distance", "fare_amount"))

   PushedFilters: [IsNotNull(trip_distance), IsNotNull(fare_amount), GreaterThan(trip_distance,0.1), GreaterThan(fare_amount,0.1)]
   ReadSchema: struct<pickup_datetime:timestamp,trip_distance:double,fare_amount:double>


In [16]:
# A predicate over a column derived mid-pipeline cannot reach a reader that has never heard
# of it: the filter stays above the scan.
print("the same query, filtered on a column derived mid-pipeline:")
scan_details(spark.read.parquet(PARQ)
             .withColumn("total_per_mile", F.col("total_amount") / F.col("trip_distance"))
             .where(F.col("total_per_mile") > 100)
             .select("pickup_datetime", "total_per_mile"))

the same query, filtered on a column derived mid-pipeline:
   PushedFilters: [IsNotNull(total_amount), IsNotNull(trip_distance)]
   ReadSchema: struct<pickup_datetime:timestamp,trip_distance:double,total_amount:double>


### Two traps in PySpark's filter syntax

In [17]:
# Python's and/or/not demand a single boolean value, which a column is not.
try:
    typed.filter((F.col("trip_time") > 60) and (F.col("fare_amount") > 0.10)).count()
    print("python 'and' -> no error (unexpected)")
except Exception as e:
    print("python 'and' ->", type(e).__name__, ":", str(e).split("\n")[0][:110])

python 'and' -> PySparkValueError : [CANNOT_CONVERT_COLUMN_INTO_BOOL] Cannot convert column into bool: please use '&' for 'and', '|' for 'or', '~'


In [18]:
# & binds more tightly than a comparison, so every comparison in a compound condition
# must be parenthesized. Without the parentheses the expression means something else.
try:
    bad = typed.filter(F.col("trip_time") > 60 & F.col("fare_amount") > 0.10)
    print("unparenthesized rows:", bad.count())
except Exception as e:
    print("unparenthesized ->", type(e).__name__, ":",
          " ".join(str(e).split())[:150])
print()
print("parenthesized rows:",
      f"{typed.filter((F.col('trip_time') > 60) & (F.col('fare_amount') > 0.10)).count():,}")

unparenthesized -> PySparkValueError : [CANNOT_CONVERT_COLUMN_INTO_BOOL] Cannot convert column into bool: please use '&' for 'and', '|' for 'or', '~' for 'not' when building DataFrame boole



parenthesized rows: 1,972,928


## Conclusion

* **The five-`substring` listing works, and that is the problem.** Its fields are strings, so
  `"10"` sorts before `"2"`; its character positions shift silently on one extra space or a
  changed format; and no amount of slicing can subtract two times.
* **One `to_timestamp` buys arithmetic, extraction and truncation.** After the conversion,
  every temporal question is a function call, and the timestamp survives a change of format.
* **Truncate to group, format only to display.** `date_format` returns strings that sort
  alphabetically, with April first.
* **The session time zone silently parameterizes all of it.** Three runs of one query disagreed
  about the busiest hour in New York. Set it explicitly, at the top of the job.
* **Built-in string functions differ from each other in ways one row never shows**:
  zero-based `getItem` against one-based `element_at`, null-propagating `concat` against
  null-skipping `concat_ws`.
* **Reduction is the cheapest transformation.** The column expression beat the Python predicate
  on the same data and the same answer, the reader pushed it into the scan, and the `select`
  removed thirteen of seventeen columns from everything downstream — while a predicate over a
  mid-pipeline derived column reached the reader not at all.

Next: [4.4](04.04%20Aggregation%20and%20Join%20Types.ipynb) spends the shuffle that reduction
was postponing.